# Week 15 Companion Notebook: Building a RAG Pipeline from Scratch
## BSAN 6200: Text Mining & Social Media Analytics -- Spring 2026

This notebook walks through every step of a **Retrieval-Augmented Generation (RAG)** pipeline.

**Design choice:** This notebook uses **minimal dependencies** and direct API calls instead of high-level LangChain wrappers. This avoids version compatibility issues and lets you see exactly what happens at each step.

| Dependency | What It Does | Install |
|---|---|---|
| `chromadb` | Vector database | `pip install chromadb` |
| `sentence-transformers` | Free local embeddings | `pip install sentence-transformers` |
| `huggingface-hub` | Free LLM (HuggingFace Inference API) | `pip install huggingface-hub` |

No `langchain`, `langchain-core`, `langchain-community`, or `langchain-huggingface` needed.

**HuggingFace token setup:** Create a token at https://huggingface.co/settings/tokens with **"Make calls to Inference Providers"** permission enabled.

---

### Table of Contents
1. Setup and Install
2. Create a Sample Corpus
3. Text Chunking
4. Embeddings and Vector Store
5. Retrieval (Similarity Search)
6. Connecting the LLM
7. Building the RAG Chain
8. Prompt Engineering
9. Tuning Retrieval: the k Parameter
10. Conversation Memory
11. Summary

---
## 1. Setup and Install

Three packages. That's it.

In [ ]:
# Run once, then comment out
# !pip install chromadb sentence-transformers huggingface-hub

In [ ]:
import os
import textwrap

# ── Your HuggingFace API token ──
# Get a free token: https://huggingface.co/settings/tokens
# IMPORTANT: When creating the token, check "Make calls to Inference Providers"
#
# Option 1: Set it directly (OK for class demos, never commit to GitHub)
# os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"

# Option 2: Load from .env file (recommended for assignments)
# from dotenv import load_dotenv
# load_dotenv()

print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))

---
## 2. Create a Sample Corpus

We create a small fictional corpus about **NovaTech**, a made-up analytics company. This lets us verify every step without needing external files.

In Assignment 5, you will replace these with your own documents (resume, cover letter, project write-ups, or job descriptions).

In [ ]:
# ── Sample documents about a fictional company ──

sample_documents = [
    {
        "text": """NovaTech Company Overview
NovaTech is a data analytics consulting firm founded in 2019 and headquartered in Los Angeles, California. 
The company specializes in helping mid-size retail and e-commerce businesses turn raw data into actionable 
insights. NovaTech employs 45 people across three departments: Data Engineering, Analytics, and Client 
Services. The company uses Python, SQL, and Tableau as its primary technology stack. In 2024, NovaTech 
was named one of the top 50 fastest-growing analytics firms in Southern California by the LA Business Journal.""",
        "source": "company_overview"
    },
    {
        "text": """NovaTech Data Engineering Team
The Data Engineering team at NovaTech is responsible for building and maintaining ETL pipelines that process 
over 2 million transactions per day for retail clients. The team uses Apache Airflow for orchestration, 
PostgreSQL and BigQuery for data warehousing, and dbt for data transformations. The team consists of 12 
engineers, led by Senior Director Maria Chen, who previously worked at Google Cloud. A typical project 
involves migrating a client from spreadsheet-based reporting to an automated data pipeline, which usually 
takes 8-12 weeks and reduces manual reporting time by 70%.""",
        "source": "data_engineering"
    },
    {
        "text": """NovaTech Analytics Services
NovaTech offers three tiers of analytics services: Basic Reporting (dashboards and KPI tracking), Advanced 
Analytics (predictive modeling, customer segmentation, churn analysis), and Strategic Consulting (C-suite 
advisory, data strategy roadmaps). The Analytics team uses scikit-learn and XGBoost for machine learning, 
along with custom NLP pipelines built with spaCy for analyzing customer feedback and support tickets. The 
most popular service is churn prediction, which has helped clients reduce customer attrition by an average 
of 18% within the first six months of deployment.""",
        "source": "analytics_services"
    },
    {
        "text": """NovaTech Internship Program
NovaTech runs a competitive summer internship program for graduate students in data science, business 
analytics, and computer science. Interns work on real client projects under mentorship from senior analysts. 
The program runs from June through August (10 weeks) and includes a stipend of $6,000 per month. Past 
interns have come from programs at USC, UCLA, LMU, and UC Irvine. Approximately 60% of interns receive 
full-time offers upon graduation. The application deadline is typically March 1st each year.""",
        "source": "internship"
    },
    {
        "text": """NovaTech Client Case Study: RetailMax
RetailMax, a regional chain of 85 home goods stores, partnered with NovaTech in 2023 to modernize its 
analytics capabilities. The engagement included migrating from Excel-based inventory tracking to an 
automated BigQuery pipeline, building a customer segmentation model that identified 5 distinct buyer 
personas, and deploying a Tableau dashboard suite used by 40 store managers daily. The project was 
completed in 14 weeks. Results after 6 months: 22% reduction in stockouts, 15% increase in targeted 
marketing ROI, and a shift from monthly to daily reporting cadence.""",
        "source": "case_study"
    },
]

print(f"Created {len(sample_documents)} sample documents")
for doc in sample_documents:
    print(f"  - {doc['source']}: {len(doc['text'])} characters")

---
## 3. Text Chunking

### Why chunk?

LLMs have a **context window** (max tokens they process at once). Chunking breaks documents into smaller pieces so you retrieve only the relevant ones instead of feeding everything in.

### The tradeoff

| Smaller chunks | Larger chunks |
|---|---|
| More precise retrieval | More surrounding context |
| May split an idea in half | May include irrelevant content |
| More chunks to search | Fewer chunks, faster search |

### Overlap

Chunks overlap by 50-100 characters so ideas at chunk boundaries are not lost.

**No LangChain needed here.** We write a simple chunker in pure Python.

In [ ]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]  # skip tiny fragments


def chunk_text_by_sentences(text, chunk_size=300, overlap=50):
    """Split text into chunks that respect sentence boundaries."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Keep overlap by taking the end of the current chunk
            words = current_chunk.split()
            overlap_text = " ".join(words[-10:]) if len(words) > 10 else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks


print("Two chunking functions defined:")
print("  chunk_text()             - fixed-size, may split mid-sentence")
print("  chunk_text_by_sentences() - respects sentence boundaries")

In [ ]:
# ── Compare the two strategies on the first document ──

test_doc = sample_documents[0]["text"]

chunks_fixed = chunk_text(test_doc, chunk_size=300, overlap=50)
chunks_sentence = chunk_text_by_sentences(test_doc, chunk_size=300, overlap=50)

print(f"Fixed-size:  {len(chunks_fixed)} chunks")
print(f"Sentence:    {len(chunks_sentence)} chunks")

print("\n" + "=" * 60)
print("FIXED-SIZE chunks:")
print("=" * 60)
for i, c in enumerate(chunks_fixed):
    print(f"\nChunk {i+1} ({len(c)} chars): {c[:100]}...")

print("\n" + "=" * 60)
print("SENTENCE-AWARE chunks:")
print("=" * 60)
for i, c in enumerate(chunks_sentence):
    print(f"\nChunk {i+1} ({len(c)} chars): {c[:100]}...")

In [ ]:
# ── Chunk ALL documents using sentence-aware strategy ──
# Each chunk keeps track of which document it came from (metadata)

all_chunks = []  # list of dicts: {"text": ..., "source": ...}

for doc in sample_documents:
    doc_chunks = chunk_text_by_sentences(doc["text"], chunk_size=400, overlap=50)
    for chunk in doc_chunks:
        all_chunks.append({
            "text": chunk,
            "source": doc["source"]
        })

print(f"Total chunks across all documents: {len(all_chunks)}")
print(f"\nChunks per source:")
from collections import Counter
for source, count in Counter(c["source"] for c in all_chunks).items():
    print(f"  {source}: {count}")

---
## 4. Embeddings and Vector Store

### What are embeddings?

An **embedding** converts text into a list of numbers (a vector) that captures meaning. Similar texts produce similar vectors. This is the same concept as Word2Vec and BERT embeddings from earlier weeks, but at the sentence/paragraph level.

### How retrieval works

1. **Indexing (one-time):** Embed every chunk and store the vectors
2. **Query time:** Embed the user's question with the same model
3. **Search:** Find the chunk vectors closest to the question vector
4. **Return:** Send the top-k closest chunks to the LLM as context

We use **sentence-transformers** for embeddings (free, local, no API key) and **ChromaDB** for the vector store (free, local).

In [ ]:
from sentence_transformers import SentenceTransformer

# Load embedding model (runs locally, no API key needed)
# First run downloads the model (~80MB), then it's cached
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# ── See what an embedding looks like ──
sample = embed_model.encode("What does NovaTech do?")
print(f"Embedding dimensions: {len(sample)}")
print(f"First 10 values: {sample[:10].tolist()}")
print(f"\nThis vector represents the MEANING of the question.")
print(f"Similar questions will produce similar vectors.")

In [ ]:
import chromadb

# ── Create a ChromaDB collection ──
chroma_client = chromadb.Client()  # in-memory, no files needed

collection = chroma_client.create_collection(
    name="novatech_demo",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity
)

# ── Add all chunks to the collection ──
collection.add(
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"]} for c in all_chunks],
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)

print(f"Vector store created with {collection.count()} chunks")
print("ChromaDB automatically embeds the text using a default model.")
print("For more control, you can pass your own embeddings (shown next).")

### Optional: Use your own embedding model with ChromaDB

By default, ChromaDB uses its own built-in embedding model. If you want to use `sentence-transformers` explicitly (for example, to match what you use in your Streamlit app), you can pass pre-computed embeddings:

```python
embeddings = embed_model.encode([c["text"] for c in all_chunks])
collection.add(
    documents=[c["text"] for c in all_chunks],
    embeddings=embeddings.tolist(),
    metadatas=[{"source": c["source"]} for c in all_chunks],
    ids=[f"chunk_{i}" for i in range(len(all_chunks))]
)
```

For this demo, the default is fine.

---
## 5. Retrieval (Similarity Search)

Before connecting the LLM, let's verify the vector store retrieves relevant chunks. This is the **retrieval** half of RAG.

In [ ]:
def search(query, k=3):
    """Search the vector store and return top-k results."""
    results = collection.query(
        query_texts=[query],
        n_results=k
    )
    # ChromaDB returns nested lists, so we unpack
    docs = results["documents"][0]
    sources = [m["source"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    return list(zip(docs, sources, distances))


# ── Test: technology question ──
query = "What technology does NovaTech use?"
print(f'Query: "{query}"\n')

for i, (text, source, dist) in enumerate(search(query, k=3)):
    print(f"Result {i+1} [source: {source}] (distance: {dist:.3f}):")
    print(f"  {text[:150]}...")
    print()

In [ ]:
# ── Test: internship question ──
query2 = "How can I get an internship?"
print(f'Query: "{query2}"\n')

for i, (text, source, dist) in enumerate(search(query2, k=2)):
    print(f"Result {i+1} [source: {source}] (distance: {dist:.3f}):")
    print(f"  {text[:150]}...")
    print()

Notice how the vector store returns chunks that are **semantically relevant** to the query, even though the query wording does not exactly match the document text. "How can I get an internship?" matches the internship document even though those exact words never appear.

If retrieval is pulling the wrong chunks, the LLM will generate wrong answers. **Retrieval quality is the foundation of RAG quality.**

---
## 6. Connecting the LLM

We use **HuggingFace Inference API** (free tier, no credit card needed).

**Setup:**
1. Create an account at https://huggingface.co
2. Go to https://huggingface.co/settings/tokens
3. Create a token with **"Make calls to Inference Providers"** checked
4. Set it as `HF_TOKEN` in your environment

If you prefer OpenAI, see the alternative at the bottom of this section.

In [ ]:
from huggingface_hub import InferenceClient

# ── Create the client ──
hf_client = InferenceClient(token=os.environ.get("HF_TOKEN"))

# We'll use this model -- free, ungated, reliable on HF Inference API
MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"

def ask_llm(prompt):
    """Send a prompt to the HuggingFace model and return the response text."""
    response = hf_client.chat_completion(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.1,
    )
    return response.choices[0].message.content

# ── Quick test ──
print("LLM test:", ask_llm("Say 'Hello, NovaTech!' in one sentence."))

### Alternative: OpenAI (paid path)

```python
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment

def ask_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content
```

### Alternative: Google Gemini (free tier)

```python
pip install google-generativeai
```

```python
import google.generativeai as genai

genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
llm = genai.GenerativeModel("gemini-2.0-flash")

def ask_llm(prompt):
    return llm.generate_content(prompt).text
```

Get a free key at https://aistudio.google.com/apikey

All three paths produce the same result. The `ask_llm()` function signature stays the same regardless of which provider you use.

---
## 7. Building the RAG Chain (No LangChain Needed)

The "chain" is just three steps:
1. **Retrieve:** search the vector store for relevant chunks
2. **Build prompt:** combine the question + retrieved chunks into a prompt
3. **Generate:** send the prompt to the LLM

That's it. No framework required.

In [ ]:
def ask_rag(question, k=3, system_prompt=None):
    """
    Complete RAG pipeline in one function:
    1. Retrieve top-k chunks from vector store
    2. Build a prompt with context + question
    3. Send to LLM and return the answer + sources
    """
    # Step 1: Retrieve
    results = search(question, k=k)
    context = "\n\n".join([text for text, source, dist in results])
    sources = [source for text, source, dist in results]
    
    # Step 2: Build prompt
    if system_prompt is None:
        system_prompt = "Use the following context to answer the question."
    
    full_prompt = f"""{system_prompt}

Context:
{context}

Question: {question}

Answer:"""
    
    # Step 3: Generate
    answer = ask_llm(full_prompt)
    
    return {
        "answer": answer,
        "sources": sources,
        "context": context,
        "prompt": full_prompt
    }


# ── Test it ──
result = ask_rag("What technology stack does NovaTech use?")

print("ANSWER:")
print(result["answer"])
print("\nSOURCES:", result["sources"])

In [ ]:
# ── Test with a question the documents CAN'T answer ──
result_oos = ask_rag("What is NovaTech's stock price?")

print("ANSWER:")
print(result_oos["answer"])
print("\nNotice: the model may hallucinate or give a vague answer.")
print("This is because the simple prompt has no instruction to say 'I don't know'.")

---
## 8. Prompt Engineering

The `system_prompt` parameter controls how the LLM behaves. This is where prompt engineering meets RAG.

### Simple prompt vs. grounded prompt

The simple prompt works for basic questions but:
- The LLM may **hallucinate** when the context doesn't have the answer
- The tone and format are unpredictable
- There is no instruction to stay grounded

Let's fix that with a **grounded prompt**.

In [ ]:
# ── Grounded prompt ──
grounded_prompt = """You are a knowledgeable assistant for NovaTech, a data analytics consulting firm.
Answer the question using ONLY the information provided in the context below.

Rules:
1. If the context does not contain enough information to answer, say: "I don't have that information in the provided documents."
2. Keep answers concise (2-4 sentences).
3. Cite which document the information comes from when possible."""

# ── Test: same questions, better prompt ──
print("=" * 50)
print("Question 1: What technology stack does NovaTech use?")
print("=" * 50)
r1 = ask_rag("What technology stack does NovaTech use?", system_prompt=grounded_prompt)
print(r1["answer"])

print("\n" + "=" * 50)
print("Question 2: What is NovaTech's stock price?")
print("=" * 50)
r2 = ask_rag("What is NovaTech's stock price?", system_prompt=grounded_prompt)
print(r2["answer"])

print("\n" + "=" * 50)
print("Question 3: Tell me about the internship program")
print("=" * 50)
r3 = ask_rag("Tell me about the internship program", system_prompt=grounded_prompt)
print(r3["answer"])

### What changed?

| Behavior | Simple Prompt | Grounded Prompt |
|---|---|---|
| Out-of-scope question | May hallucinate | Says "I don't have that information" |
| Answer length | Unpredictable | Concise (2-4 sentences) |
| Source attribution | None | Cites the document |
| Tone | Generic | Professional, consistent |

In Assignment 5, you will design and iterate on your own prompt. The rubric requires at least 3 documented iterations.

---
## 9. Tuning Retrieval: the `k` Parameter

`k` controls how many chunks are retrieved. This is a key lever:

| Small k (1-2) | Large k (4-6) |
|---|---|
| Focused, less noise | More context, more complete |
| May miss relevant info | May include irrelevant chunks |
| Faster, cheaper | Slower, more tokens used |

In [ ]:
# ── Compare k=1 vs k=3 vs k=5 ──
query = "What services does NovaTech offer and what results have they achieved?"

for k_val in [1, 3, 5]:
    result = ask_rag(query, k=k_val, system_prompt=grounded_prompt)
    print(f"\n{'=' * 50}")
    print(f"k={k_val} | Sources: {result['sources']}")
    print(f"{'=' * 50}")
    print(result["answer"])

class RAGChatbot:
    """RAG chatbot with conversation memory."""
    
    def __init__(self, collection, system_prompt, k=3):
        self.collection = collection
        self.system_prompt = system_prompt
        self.k = k
        self.history = []  # list of {"role": "user"/"assistant", "content": "..."}
    
    def ask(self, question):
        # Step 1: Retrieve relevant chunks
        results = self.collection.query(query_texts=[question], n_results=self.k)
        context = "\n\n".join(results["documents"][0])
        sources = [m["source"] for m in results["metadatas"][0]]
        
        # Step 2: Build prompt with history + context
        history_text = ""
        if self.history:
            history_text = "\nPrevious conversation:\n"
            for msg in self.history[-6:]:  # keep last 3 exchanges
                role = "User" if msg["role"] == "user" else "Assistant"
                history_text += f"{role}: {msg['content']}\n"
        
        full_prompt = f"""{self.system_prompt}
{history_text}
Context from documents:
{context}

User question: {question}

Answer:"""
        
        # Step 3: Generate
        answer = ask_llm(full_prompt)
        
        # Step 4: Update history
        self.history.append({"role": "user", "content": question})
        self.history.append({"role": "assistant", "content": answer})
        
        return {"answer": answer, "sources": sources}
    
    def reset(self):
        self.history = []


print("RAGChatbot class defined")

# ── Multi-turn conversation ──
chatbot = RAGChatbot(
    collection=collection,
    system_prompt=grounded_prompt,
    k=3
)

questions = [
    "What does NovaTech do?",
    "How big is the team?",
    "Do they have internships?",
    "What is the stipend for that?",   # "that" refers to the internship
]

for q in questions:
    result = chatbot.ask(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}\n")

In [ ]:
class RAGChatbot:
    """RAG chatbot with conversation memory."""
    
    def __init__(self, collection, llm, system_prompt, k=3):
        self.collection = collection
        self.llm = llm
        self.system_prompt = system_prompt
        self.k = k
        self.history = []  # list of {"role": "user"/"assistant", "content": "..."}
    
    def ask(self, question):
        # Step 1: Retrieve relevant chunks
        results = self.collection.query(query_texts=[question], n_results=self.k)
        context = "\n\n".join(results["documents"][0])
        sources = [m["source"] for m in results["metadatas"][0]]
        
        # Step 2: Build prompt with history + context
        history_text = ""
        if self.history:
            history_text = "\nPrevious conversation:\n"
            for msg in self.history[-6:]:  # keep last 3 exchanges
                role = "User" if msg["role"] == "user" else "Assistant"
                history_text += f"{role}: {msg['content']}\n"
        
        full_prompt = f"""{self.system_prompt}
{history_text}
Context from documents:
{context}

User question: {question}

Answer:"""
        
        # Step 3: Generate
        response = self.llm.generate_content(full_prompt)
        answer = response.text
        
        # Step 4: Update history
        self.history.append({"role": "user", "content": question})
        self.history.append({"role": "assistant", "content": answer})
        
        return {"answer": answer, "sources": sources}
    
    def reset(self):
        self.history = []


print("RAGChatbot class defined")

In [ ]:
# ── Multi-turn conversation ──
chatbot = RAGChatbot(
    collection=collection,
    llm=llm,
    system_prompt=grounded_prompt,
    k=3
)

questions = [
    "What does NovaTech do?",
    "How big is the team?",
    "Do they have internships?",
    "What is the stipend for that?",   # "that" refers to the internship
]

for q in questions:
    result = chatbot.ask(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}\n")

Notice that the last question ("What is the stipend for **that**?") works because the memory stores the conversation history. Without memory, the model would not know what "that" refers to.

For Assignment 5, you can choose whether to use stateless (simpler) or add memory (more polished chatbot).

---
## 11. Streamlit Chat Interface (Overview)

Streamlit cannot run inside a Jupyter notebook. This section shows the **key building blocks** for your `streamlit_app.py`.

```python
import streamlit as st
import chromadb
from huggingface_hub import InferenceClient

# ── Load resources once ──
@st.cache_resource
def load_vectorstore():
    client = chromadb.PersistentClient(path="./chroma_db")
    return client.get_collection("my_collection")

@st.cache_resource  
def load_llm():
    return InferenceClient(token=st.secrets["HF_TOKEN"])

collection = load_vectorstore()
hf_client = load_llm()

# ── Chat interface ──
st.title("My RAG Chatbot")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

user_input = st.chat_input("Ask me something...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)
    
    # Your RAG logic here (retrieve -> build prompt -> generate)
    answer = "Replace with your ask_rag() call"
    
    with st.chat_message("assistant"):
        st.markdown(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})
```

### Key patterns

| Pattern | What It Does |
|---|---|
| `@st.cache_resource` | Load expensive resources (vector store, LLM) only once |
| `st.session_state` | Persist chat history across reruns |
| `st.chat_input` / `st.chat_message` | Chat-style UI widgets |
| `st.expander` | Collapsible section for showing retrieved chunks |
| `st.secrets` | Store API keys safely (never hardcode) |

Run your app with: `streamlit run streamlit_app.py`

---
## 12. Summary: The Full RAG Pipeline

```
YOUR DOCUMENTS
    |
    v
[1] Chunk text              -- chunk_text_by_sentences()
    |
    v
[2] Embed chunks            -- sentence-transformers (free, local)
    |
    v
[3] Store in vector DB      -- ChromaDB
    |
    v
[4] User asks a question
    |
    v
[5] Search vector store     -- collection.query()
    |
    v
[6] Build prompt            -- context + question + system instructions
    |
    v
[7] LLM generates answer   -- HuggingFace (free) or OpenAI (paid)
    |
    v
[8] Return answer + sources
```

### What we used (3 packages total)

| Package | Purpose | Cost |
|---|---|---|
| `sentence-transformers` | Embeddings | Free (local) |
| `chromadb` | Vector database | Free (local) |
| `huggingface-hub` | LLM (Inference API) | Free tier |

### For Assignment 5

- **Option A:** Replace NovaTech docs with your resume, cover letter, project write-ups
- **Option B:** Replace with 10+ job descriptions and your resume, build comparison prompts
- The RAG infrastructure (steps 1-3, 5-8) stays the same. What changes is the **documents** and the **prompts**.

### What about LangChain?

LangChain is a popular framework that wraps all these steps into higher-level abstractions. It can speed up prototyping but has well-documented drawbacks: frequent breaking changes across versions, dependency bloat, and abstractions that hide what is happening at each step.

For learning, building RAG from scratch (as we did here) gives you a clearer understanding. You are welcome to use LangChain in your assignment if you prefer, but it is not required.

### What about the LLM provider?

This notebook uses HuggingFace Inference API (free). You can swap to any provider by changing the `ask_llm()` function. The rest of the pipeline stays identical. See Section 6 for OpenAI and Gemini alternatives.

---
*BSAN 6200 | Spring 2026 | Week 15 Companion Notebook*